# A basic model

This page builds the smallest useful epidemic model, an SIR model, and runs it. By the end
you will have declared compartments, added an infection flow and a recovery flow, set the
starting population, run the model for 100 days and plotted the result.

The model has three compartments: susceptible (S), infectious (I) and recovered (R). People
move from S to I when they are infected, and from I to R when they recover. The infection
flow depends on how many people are currently infectious. The recovery flow does not.

```{admonition} Attribution
:class: note

Parts of this page's prose are adapted from the
[summer2 documentation](https://summer2.readthedocs.io), BSD-2-Clause,
Copyright (c) 2022, monash-emu
([monash-emu/summer2](https://github.com/monash-emu/summer2)). The code is
written for summer4.
```

## Imports

The next cell loads the libraries. summer4 does the modelling, pandas holds the results and
Plotly draws the charts. Nothing is drawn yet.

In [ ]:
import numpy as np
import pandas as pd
import plotly.io as pio

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

from summer4 import (
    Compartments,
    FlowModel,
    Param,
    Property,
    PropertyMap,
    SavePlan,
    SaveRequest,
    TransitionFlow,
)
from summer4.epi import ForceOfInfection

## Compartments

A `Property` is a named list of the values a person can have. Here the property is the
disease `state` and its values are `S`, `I` and `R`. A `PropertyMap` turns the property into
the list of compartments the model tracks. The cell prints the compartment labels.

In [ ]:
state = Property("state", ("S", "I", "R"))
pmap = PropertyMap.from_property(state)
pmap.labels()

## Flows

A `FlowModel` collects flows. A `TransitionFlow` moves people from one compartment to
another: `TransitionFlow(name, source, destination, rate)`.

The recovery rate is a per-person rate, so 0.2 per day means that a fifth of the infectious
people recover each day. We write it as a `Param` so that we can change it when we run the
model without rebuilding the model.

The infection rate is the **force of infection**: the contact rate times the proportion of
the population that is infectious, `contact_rate * I / N`. `ForceOfInfection` builds exactly
that. It names the infectious compartments and takes the contact rate, which is also a
`Param`. The cell builds the model and prints nothing.

In [ ]:
model = FlowModel(pmap)
model.add_flow(
    TransitionFlow(
        "infection",
        state["S"],
        state["I"],
        ForceOfInfection("infection", infectious=state["I"], contact_rate=Param("contact_rate")),
    )
)
recovery_flow = model.add_flow(
    TransitionFlow("recovery", state["I"], state["R"], Param("recovery_rate"))
)

## Starting population

`set_initial_population` says how many people start in each compartment. Here 990 people are
susceptible and 10 are infectious. Anyone not listed starts with zero.

In [ ]:
start = model.set_initial_population({state["S"]: 990.0, state["I"]: 10.0})

## Compile and run

`compile()` turns the model into something that can be run quickly. We compile once and can
then run it as often as we like.

A `SavePlan` says what to keep from the run. `Compartments()` keeps the number of people in
every compartment at every saved time. `run` takes the parameter values, a start time `t0`,
an end time `t1` and a step size `dt`, which are all in days. We save the result once per
day.

In [ ]:
compiled = model.compile()
plan = SavePlan(
    requests={"compartments": SaveRequest(Compartments())},
    ts=np.linspace(0.0, 100.0, 101),
)
result = compiled.run(
    {"contact_rate": 0.5, "recovery_rate": 0.2}, t0=0.0, t1=100.0, dt=0.1, save=plan
)

## Plot the compartments

`result["compartments"]` is the saved output. `to_pandas()` turns it into a table with one
column per compartment. The plot should show the susceptible line falling, the infectious
line rising to a peak of about 240 people around day 17 and then falling back to nearly zero, and
the recovered line rising to about 900 people.

In [ ]:
frame = result["compartments"].to_pandas()
frame.plot(
    title="SIR model: people in each compartment",
    labels={"index": "time (days)", "value": "people"},
)

## Run it again with different parameters

Because the contact rate is a `Param`, we can run the compiled model again with a new value.
The plot compares the infectious compartment for three contact rates. A higher contact rate
should give a taller, earlier peak.

In [ ]:
infectious = {}
for contact_rate in (0.3, 0.5, 0.8):
    run = compiled.run(
        {"contact_rate": contact_rate, "recovery_rate": 0.2},
        t0=0.0,
        t1=100.0,
        dt=0.1,
        save=plan,
    )
    infectious[f"contact rate {contact_rate}"] = (
        run["compartments"].select(state["I"]).to_pandas().iloc[:, 0]
    )
pd.DataFrame(infectious).plot(
    title="Infectious people for three contact rates",
    labels={"index": "time (days)", "value": "infectious people"},
)

## Things to try

- Change the recovery rate to 0.1 and rerun. How does the peak change?
- Seed the epidemic with 1 infectious person instead of 10. What happens to the timing of the
  peak?
- Add a fourth compartment, `E` for exposed, to the `Property`, and add a flow from `E` to
  `I`. Send infection from `S` to `E` instead of `S` to `I`.
- Print `frame.sum(axis=1)`. Does the total number of people change over time? Why not?